# Community Detection in Social Networks
### Course: Pattern Recognition
**Author:** Aditya Mahalle (USN: CM23054)

---
## Project Overview & Real-World Datasets
We evaluate three fundamental community detection algorithms on standard real-world social benchmarks:
1. **American College Football Network** (115 teams across 12 athletic conferences - Girvan & Newman 2002)
2. **Facebook Social Circles** (78 users across 4 social circles - SNAP)
3. **Academic Co-Authorship Network** (120 researchers across 3 scientific domains - DBLP)

---
## 5-Step Pipeline:
1. **Step 1: Collect Data** (Load real-world social graphs with ground-truth conference/circle labels)
2. **Step 2: Build & Clean Graph** (NetworkX graph construction, remove self-loops & isolates)
3. **Step 3: Detect Communities** (Louvain, Girvan-Newman, Label Propagation)
4. **Step 4: Evaluate** (Modularity Q, Conductance, NMI, ARI, Execution Time)
5. **Step 5: Visualize & Interpret** (2D colored community maps & Gephi GEXF exports)


## Step 1 & 2: Load & Clean the American College Football Network
115 college football teams with regular season games split across 12 conferences.

In [ ]:
import time
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from datasets import get_football_network, get_facebook_circles, get_coauthorship_network

# Load Football Network
G, info = get_football_network()
ground_truth = info['ground_truth']

# Clean graph
G.remove_edges_from(nx.selfloop_edges(G))
G.remove_nodes_from(list(nx.isolates(G)))

print(f"Loaded: {info['name']}")
print(f"Nodes (Teams): {G.number_of_nodes()}")
print(f"Edges (Games): {G.number_of_edges()}")
print(f"Ground-Truth Conferences: {info['ground_truth_k']}")

## Step 3: Run the 3 Community Detection Algorithms

In [ ]:
# 1. Louvain Algorithm (Modularity Optimization)
t0 = time.time()
louvain_comms = list(nx.community.louvain_communities(G, seed=42))
louvain_time = (time.time() - t0) * 1000
louvain_partition = {node: cid for cid, members in enumerate(louvain_comms) for node in members}

# 2. Girvan-Newman Algorithm (Edge Betweenness Divisive)
t0 = time.time()
gn_generator = nx.community.girvan_newman(G)
best_gn_comms = None
for step, comm_tuple in enumerate(gn_generator):
    clist = [sorted(list(c)) for c in comm_tuple]
    if len(clist) >= info['ground_truth_k']:
        best_gn_comms = clist
        break
    if step >= 15: break
if not best_gn_comms: best_gn_comms = clist
gn_time = (time.time() - t0) * 1000
gn_partition = {node: cid for cid, members in enumerate(best_gn_comms) for node in members}

# 3. Label Propagation Algorithm (Linear Time Diffusion)
t0 = time.time()
lpa_comms = list(nx.community.asyn_lpa_communities(G, seed=42))
lpa_time = (time.time() - t0) * 1000
lpa_partition = {node: cid for cid, members in enumerate(lpa_comms) for node in members}

print("Execution finished for Louvain, Girvan-Newman, and Label Propagation.")

## Step 4: Evaluation & Benchmark Matrix

In [ ]:
def evaluate_algo(G, partition, comms, name, exec_time, ground_truth=None):
    q = nx.community.modularity(G, [set(c) for c in comms])
    conds = [nx.conductance(G, set(c)) for c in comms if 0 < len(c) < G.number_of_nodes()]
    avg_cond = np.mean(conds) if conds else 0.0
    nmi, ari = None, None
    if ground_truth:
        nodes = sorted(list(G.nodes()))
        y_pred = [partition[n] for n in nodes]
        y_true = [ground_truth[n] for n in nodes]
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ari = adjusted_rand_score(y_true, y_pred)
    return {
        'Algorithm': name,
        'Communities': len(comms),
        'Modularity Q': round(q, 4),
        'Conductance': round(avg_cond, 4),
        'NMI Accuracy': round(nmi, 4) if nmi else 'N/A',
        'ARI': round(ari, 4) if ari else 'N/A',
        'Time (ms)': round(exec_time, 2)
    }

df_results = pd.DataFrame([
    evaluate_algo(G, louvain_partition, louvain_comms, 'Louvain', louvain_time, ground_truth),
    evaluate_algo(G, gn_partition, best_gn_comms, 'Girvan-Newman', gn_time, ground_truth),
    evaluate_algo(G, lpa_partition, lpa_comms, 'Label Propagation', lpa_time, ground_truth)
])
display(df_results)

## Step 5: Visualizing the Community Map

In [ ]:
plt.figure(figsize=(10, 8))
palette = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2', '#59A14F', '#EDC948', '#B07AA1', '#FF9DA7', '#9C755F', '#BAB0AC', '#86BCB6', '#D37295']
node_colors = [palette[louvain_partition[n] % len(palette)] for n in G.nodes()]
pos = nx.spring_layout(G, seed=42)

nx.draw_networkx_edges(G, pos, alpha=0.25, edge_color='gray')
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=350, edgecolors='black')
nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold')
plt.title(f"American College Football - Louvain Communities (NMI = {df_results.loc[0, 'NMI Accuracy']})", fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()